# Dedicated E2M1/A32 - honderd epochs

CNN-8 / SVHN train-only. Rechtstreeks E2M1 kwantiseren; FP32-master; een forward/backward; geen distillatie, teacher of truncatie. Dit notebook leest alleen opgeslagen rapporten. Beste test-geselecteerde checkpoint en vast eindpunt blijven gescheiden.

In [1]:
from pathlib import Path
RUN_DIR = Path.cwd()
CONTROL_DIR = None

In [2]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def read(name):
    return json.loads((RUN_DIR / name).read_text(encoding='utf-8'))
R = read('run_report.json'); H = read('history.json'); CFG = R['config']
assert CFG['weight_format'] == 'MXFP4_E2M1' and not CFG['truncation'] and not CFG['distillation']
assert CFG['activation_bits'] == 32 and CFG['train_split'] == 'train' and not CFG['extra_used']
E = pd.read_csv(RUN_DIR / 'epoch_metrics.csv')
T = pd.read_csv(RUN_DIR / 'epoch_times.csv')
SEL = pd.read_csv(RUN_DIR / 'selected_metrics.csv')
assert not E.duplicated(['epoch','phase']).any()
assert len(E) == R['completed_epoch'] * 2
assert ((E.count_top1_percent - 100*E.correct1/E.n).abs() < 1e-8).all()
SYN = bool(R.get('synthetic_fixture_only', False))
if SYN:
    display(Markdown('## LET OP: uitsluitend synthetische softwarevalidatie, geen onderzoeksresultaten.'))
display(pd.DataFrame([{'Status':R['status'],'Epochs':R['completed_epoch'], 'Updates':R['optimizer_steps'],
                      'Formaat':CFG['weight_format'], 'Activaties':'A32', 'Synthetisch':SYN}]))


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\renzo\\master-thesis-mx\\software_checks\\mx_e2m1_dedicated_reference100_v1\\run_report.json'

## 1. Beste en laatste checkpoint
Hoofdvergelijking voor de meeting: epoch 100 met epoch 100 uit de andere trainingen. De beste-scoreselectie gebruikt de testset en is geen onafhankelijke testschatting. Een voortijdig gestopte run is geen volledige honderd-epochreferentie.

In [ ]:
table = SEL[['role','epoch','correct1','n','count_top1_percent','objective_loss']].rename(columns={
    'role':'Checkpointrol','epoch':'Epoch','count_top1_percent':'Accuracy (%)','objective_loss':'Test-CE'})
display(table.style.hide(axis='index').format({'Accuracy (%)':'{:.4f}','Test-CE':'{:.4f}'}))
assert all(r['test'][0]['weight_format']=='MXFP4_E2M1' for r in H['epochs'])


## 2. Leercurves

In [ ]:
for metric, ylabel in [('count_top1_percent','Accuracy (%)'),('objective_loss','Cross-entropy')]:
    fig, ax = plt.subplots(figsize=(9,4.4))
    for phase, label in [('train','Training'),('test','Test')]:
        q=E[E.phase==phase]; ax.plot(q.epoch,q[metric],label=label)
    for epoch in [50,75,90]: ax.axvline(epoch,linestyle=':',alpha=.4)
    ax.set(xlabel='Afgeronde epoch',ylabel=ylabel,title='Dedicated E2M1/A32'+(' - synthetische test' if SYN else ''))
    ax.legend();ax.grid(alpha=.2);fig.tight_layout();plt.show();plt.close(fig)


## 3. Werkelijke looptijd en LR
Emulatietijd inclusief registratie; geen bewijs van hardwareversnelling. De vijf-epochproeven en eerdere zesbitslooptijden zijn geen gemeten E2M1-looptijd.

In [ ]:
elapsed = float(T.total_epoch_seconds.sum())
display(pd.DataFrame([{'Epochs':len(T),'Gemeten epochs (min)':elapsed/60,
                     'Sessie (min)':R.get('session_wall_seconds',elapsed)/60,
                     'Volledige training':R['full_training_completed']}]))
train=E[E.phase=='train']; display(train[['epoch','lr']].drop_duplicates('lr'))
fig,ax=plt.subplots(figsize=(9,3.5));ax.plot(T.epoch,T.total_epoch_seconds)
ax.set(xlabel='Epoch',ylabel='Seconden',title='Gemeten epochtijd');ax.grid(alpha=.2)
fig.tight_layout();plt.show();plt.close(fig)


## 4. Integriteit en exports
De zes gewichtsmasters, gradients en Adam-momenten blijven FP32. Alleen BN32 leert. De export bevat directe E2M1-codes plus schalen en bewaarde FP32-uitzonderingslagen/BN-staat, geen hoge E2M3-bron. Er vindt geen extra volledige testevaluatie van de export plaats in deze runner.

In [ ]:
assert all(x['gradient_checks']['finite_gradient_checks']==x['gradient_checks']['optimizer_steps_in_phase'] for x in H['epochs'])
assert all(x['test_optimizer_steps']==0 and x['batchnorm_counters_correct'] for x in H['epochs'])
display(pd.DataFrame(R.get('warnings',[])))
rows=[]
for suffix in ['best','last']:
    p=RUN_DIR/f'mx_export_layers_{suffix}.csv'
    if p.is_file():
        q=pd.read_csv(p); rows.append({'rol':suffix,'gewichtcodes + schalen (B)':int(q.packed_weight_and_scale_bytes.sum()),
            'gewichten':int(q.n_weights.sum()),'blokken':int(q.blocks.sum())})
display(pd.DataFrame(rows))


## 5. Interpretatie
Een dedicated model ondersteunt hier slechts een precisie. De vergelijking met gezamenlijk getrainde modellen vergelijkt trainingsregimes: ook loss, teacher en aantal passes verschillen. Geen garantie dat dedicated nauwkeuriger is. Een seed is geen bewijs van statistische superioriteit.

Voor integratie in het bestaande overzicht: gebruik `voeg_toe_aan_overzicht.py` uit het codepakket, of stuur de feedbackbundel voor de gecontroleerde uitbreiding van de meetingnotebook. De bestaande bronbestanden blijven ongewijzigd.